# 12 - Drift monitoring (alerts for human review, never automatic retraining)
Spec section 16. Seven signals per frame, computed exactly as the controller sees them (notebook 10, Standard mode, YOLO11n): **brightness** (image darkness), **confidence** (detection uncertainty), **density**, **occlusion**, **instability**, **difficulty D**, **global slot risk R**.

1. **Baseline** per camera: 101 quantiles of each signal over the camera's val frames (the same compact baseline runs on the Orin). For val windows the camera's own day is left out of its baseline.
2. **Current**: a rolling window of the camera's last `WINDOW` frames (8 frames = 40 min on PKLot, 4 h on CNRPark-EXT). Score per signal: **PSI** or **KS**.
3. **Thresholds from clean val windows**: per-signal warning level so that about 5% of clean windows warn on any signal; critical = the overall level exceeded by 1% of clean windows. Rates are set, not tuned for a result.
4. **Synthetic drift**: in sampled sequences the second half of the frames is perturbed (dark, haze, blur, noise, JPEG, camera shift; mild and strong) and **re-detected with YOLO11n**. Measured: alert rate of windows entirely after onset, alert rate before onset, first-alert delay in frames, and the **real slot-accuracy drop** of the perturbed frames, so an alert can be compared with actual degradation. PSI vs KS is chosen on val (alert rate on drifted windows, then delay); test reported once.
5. **Natural drift**: test days against the val baseline, without perturbation: how often weather / season changes raise alerts.

Alerts carry severity, the signals responsible, a plain-language explanation and an acknowledgement flag. Distribution shift is not proof of accuracy loss; nothing is retrained.

**Inputs (Add Input -> Your Work):** notebook 03 output (images + splits), notebook 04 output (YOLO11n weights), notebook 07 output (full run), notebook 09 output (full run).
**Accelerator:** GPU T4 (YOLO11n on the perturbed frames). Internet ON (repo + pinned ultralytics).
**How to run:** `SMOKE = True` + Run All. Then `SMOKE = False` + **Save Version -> Save & Run All**.

In [ ]:
# --- Setup: get the project code onto Kaggle ---
# Preferred: attach the code as a Kaggle dataset (works for a private GitHub repo).
# Fallback: git clone (only works if the GitHub repo is public and Internet is ON).
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/Amith2277/Edge-Computing-EndSem.git"
REPO_DIR = "/kaggle/working/repo"
MARK = os.path.join("src", "common", "config.py")

def find_attached_code():
    for pat in ("/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"):
        for d in sorted(glob.glob(pat)):
            if os.path.isfile(os.path.join(d, MARK)):
                return d.rstrip("/")
    return None

if os.path.isdir(REPO_DIR):
    print("using existing", REPO_DIR)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):  # cloned earlier: fetch the latest code
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout.strip())
else:
    attached = find_attached_code()
    if attached:
        shutil.copytree(attached, REPO_DIR)
        print("copied attached code dataset:", attached)
    else:
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout)
        if r.returncode != 0:
            raise SystemExit("No attached code dataset found and git clone failed (private repo?). "
                             "Upload adaptivepark-code.zip as a Kaggle dataset and Add Input it, then re-run.")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("code at", REPO_DIR, "| files:", sorted(os.listdir(REPO_DIR))[:12])


In [ ]:
# Install the pinned ultralytics (same version as the Orin so the .pt loads there). No torch upgrade.
import importlib.metadata as md_
try:
    have = md_.version("ultralytics")
except md_.PackageNotFoundError:
    have = None
if have != "8.3.40":
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.3.40"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    print(r.stdout[-2000:])
# torch >= 2.6 defaults torch.load(weights_only=True), which ultralytics 8.3.40 checkpoints do not satisfy.
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")
# Training uses one GPU. Hiding the second one lets ultralytics use all 4 CPU cores for data loading
# (it divides the cores by the number of visible GPUs: 2 loader workers on "T4 x2"). Must be set before torch loads.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import torch, ultralytics
# Turn off experiment-tracker integrations (Kaggle's ray breaks the raytune callback; wandb can wait for a login).
# Runs before any trainer exists. If you saw a raytune error earlier, restart the kernel first.
from src.training import yolo_train as yt
print("tracker integrations:", yt.disable_integrations())
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
# Settings and inputs
SMOKE = True                      # True: few sequences. False: all sequences for baselines, N_SYNTH per split perturbed
SMOKE_SEQUENCES = 8
N_SYNTH = 12                      # sequences per split that get synthetic drift (stratified by site)
WINDOW = 8                        # rolling window, frames
MIN_LEN = 24                      # a synthetic-drift sequence needs this many frames (onset at the middle)
WARN_RATE, CRIT_RATE = 0.05, 0.01 # share of clean val windows allowed to warn / reach critical
LIGHT = "yolo11n"
CHUNK_FRAMES = 16                 # frames loaded at once; every variant of them is detected in one call
MAX_GAP_FACTOR = 3.0
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"

import os, json, time, csv, gzip, pickle, gc
from collections import defaultdict, Counter
from datetime import datetime
from pathlib import Path
import numpy as np
from PIL import Image
from src.common.config import load_yaml, config_hash
from src.common.status import new_run_id, write_status
from src.common.hashing import sha256_file
from src.training import yolo_train as yt
from src.evaluation import det_eval as de
from src.occupancy import association as asc
from src.temporal import filters as tf
from src.scene_analysis import features as sf
from src.slot_risk import risk as rk
from src.controller import replay as rp
from src.drift import monitor as dm, perturb as pt
DCFG = load_yaml("configs/dataset.yaml"); FRAC = DCFG["dataset"]["subset_fraction"]
IMGSZ = load_yaml("configs/train_{}.yaml".format(LIGHT))["imgsz"]

mans = [p for p in yt.find_files(INPUT_ROOT, "split_manifest.csv") if p.parent.name == "splits"]
assert mans, "split_manifest.csv not found: attach notebook 03's output"
ROW_BY_ID = {int(r["image_id"]): r for r in de.read_manifest(mans[0])}
SPLIT_INFO = json.loads((mans[0].parent / "split_info.json").read_text())
EXPORT = yt.find_export_dir(INPUT_ROOT, FRAC)
occ = [p.parent for p in yt.find_files(INPUT_ROOT, "slot_scores_test.csv.gz") if p.parent.name == "occupancy"]
assert occ, "attach notebook 07's output (full run)"
OCC = occ[0]; st07 = json.loads((OCC / "status.json").read_text()); assert st07["state"] == "COMPLETE"
rec = json.loads((OCC / "occupancy_record_{}.json".format(LIGHT)).read_text())
assert not rec["smoke_test"] and rec["dataset_split_hash"] == SPLIT_INFO["manifest_hash"]
THR, ASSOC = float(rec["chosen_on_val"]["conf_thr"]), rec["chosen_on_val"]
recs09 = [p for p in yt.find_files(INPUT_ROOT, "scene_risk_record.json") if p.parent.name == "scene_risk"]
assert recs09, "attach notebook 09's output (full run)"
SCN = recs09[0].parent; R09 = json.loads(recs09[0].read_text())
assert not R09["smoke_test"] and R09["dataset_split_hash"] == SPLIT_INFO["manifest_hash"] and R09["notebook07_run_id"] == st07["run_id"]
ALL_TREC = yt.find_files(INPUT_ROOT, "training_record.json")
found = [p for p in ALL_TREC if p.parent.name == "{}_f{}".format(LIGHT, FRAC)]
assert found, ("YOLO11n weights not found: Add Input -> Your Work -> notebook 04 (train YOLO11n) output, the same one notebook 07 used. "
               "Looked for a folder named {}_f{} holding training_record.json + best.pt; training records found: {}").format(
               LIGHT, FRAC, [str(p.parent) for p in ALL_TREC] or "none")
TREC = json.loads(found[0].read_text()); WEIGHTS = found[0].parent / "best.pt"
assert sha256_file(WEIGHTS) == TREC["checkpoint_sha256"] and TREC["dataset_split_hash"] == SPLIT_INFO["manifest_hash"]
SCALES = R09["difficulty"]["scales"]
CTX = {"thr": {LIGHT: THR}, "wD": np.array([R09["difficulty"]["weights"][f] for f in sf.FEATURES]),
       "wR": np.array([R09["slot_risk"]["weights"][c] for c in rk.COMPONENTS]), "agg": R09["slot_risk"]["aggregation"],
       "age_ref_s": float(R09["slot_risk"]["max_age_s_for_A"]), "inst_frames": int(R09["instability_frames"]),
       "occ_scale": float(SCALES["occlusion"]), "light": LIGHT, "heavy": LIGHT}
SLOTS = {s["slot_id"]: s for s in json.loads((OCC / "slots.json").read_text())}
REGION = {}
for s in SLOTS.values():
    xs, ys = s["polygon"][0::2], s["polygon"][1::2]
    b = REGION.get(s["camera"], [1e9, 1e9, -1e9, -1e9])
    REGION[s["camera"]] = [min(b[0], min(xs)), min(b[1], min(ys)), max(b[2], max(xs)), max(b[3], max(ys))]
import torch
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("images", EXPORT, "| weights", WEIGHTS, "| score threshold", THR, "| device", DEVICE, "| window", WINDOW, "frames")
OUT = Path(WORK) / "artifacts" / "drift"
OUT.mkdir(parents=True, exist_ok=True)
SUF = "_smoke" if SMOKE else ""
CFG_HASH = config_hash({"smoke": SMOKE, "smoke_sequences": SMOKE_SEQUENCES, "n_synth": N_SYNTH, "window": WINDOW, "min_len": MIN_LEN,
                        "rates": [WARN_RATE, CRIT_RATE], "split": SPLIT_INFO["manifest_hash"], "nb07": st07["run_id"], "nb09": R09["run_id"],
                        "weights": TREC["checkpoint_sha256"]})
RUN_ID = new_run_id("drf")
STATUS = OUT / "status.json"
write_status(STATUS, "12_drift_monitoring", RUN_ID, "RUNNING", CFG_HASH, "inputs loaded")

In [ ]:
# 1-3. Clean signals of every sequence (cached YOLO11n detections), per-camera baselines, thresholds from clean val windows
def parse_ts(s):
    return datetime.fromisoformat(s.replace("Z", "")).timestamp()

def load_scores(split):
    cols = defaultdict(list)
    with gzip.open(str(OCC / "slot_scores_{}.csv.gz".format(split)), "rt") as f:
        for r in csv.DictReader(f):
            for k in ("image", "camera", "timestamp", "slot_id"):
                cols[k].append(r[k])
            cols["image_id"].append(int(r["image_id"])); cols["gt"].append(int(r["gt_occupied"])); cols["valid"].append(int(r["valid"]))
            cols["score"].append(float(r["score_" + LIGHT]))
    return {k: np.array(x) for k, x in cols.items()}

PREP = {}
def prep(sids):
    if sids not in PREP:
        PREP[sids] = asc.prepare_slots([SLOTS[s]["polygon"] for s in sids])
    return PREP[sids]

def frame_inputs(sids, dets):
    pairs = asc.image_pairs(prep(sids), dets)
    return {"score": asc.slot_scores(len(sids), pairs, ASSOC["method"], ASSOC.get("overlap_thr"), ASSOC["exclusive"]),
            "detf": sf.detection_features(dets, THR, len(sids)), "occ": rk.slot_occlusion(len(sids), pairs, THR, float(ASSOC["overlap_thr"]))}

def build(split):
    o = load_scores(split)
    with gzip.open(str(SCN / "frame_features_{}.csv.gz".format(split)), "rt") as f:
        ff = {r["image"]: r for r in csv.DictReader(f)}
    z = np.load(str(OCC / "detections" / "{}_{}.npz".format(LIGHT, split)))
    drow = {str(n): i for i, n in enumerate(z["names"])}
    names, first, counts = np.unique(o["image"], return_index=True, return_counts=True)
    fr = np.argsort(first); names, first, counts = names[fr], first[fr], counts[fr]
    cam = o["camera"][first]; iid = o["image_id"][first]
    sec = np.array([parse_ts(t) for t in o["timestamp"][first]]); day = np.array([ROW_BY_ID[int(i)]["day"] for i in iid])
    sp = tf.split_sequences(cam, sec, day, MAX_GAP_FACTOR)
    cam_cols = defaultdict(dict)
    for c, s in zip(o["camera"], o["slot_id"]):
        cam_cols[c].setdefault(s, len(cam_cols[c]))
    ids = sorted(set(sp["seq"].tolist()))
    if SMOKE:
        ids = [ids[i] for i in np.linspace(0, len(ids) - 1, min(SMOKE_SEQUENCES, len(ids))).astype(int)]
    seqs = []
    for q in ids:
        fidx = np.nonzero(sp["seq"] == q)[0]; fidx = fidx[np.argsort(sec[fidx], kind="stable")]
        c = cam[fidx[0]]; r0 = ROW_BY_ID[int(iid[fidx[0]])]
        S = {"camera": c, "site": r0["site"], "day": day[fidx[0]], "sec": sec[fidx], "n_cols": len(cam_cols[c]), "names": [str(names[f]) for f in fidx],
             "image_ids": iid[fidx], "climate": [ROW_BY_ID[int(iid[f])].get("climate") or "unknown" for f in fidx],
             "cols": [], "valid": [], "gt": [], "slot_ids": [], "B": [], "M": [], "fixable": np.zeros(len(fidx)), "out": {LIGHT: defaultdict(list)}}
        for f in fidx:
            a, b = first[f], first[f] + counts[f]
            sids = tuple(o["slot_id"][a:b])
            S["cols"].append(np.array([cam_cols[c][s] for s in sids])); S["valid"].append(o["valid"][a:b].astype(bool))
            S["gt"].append(o["gt"][a:b]); S["slot_ids"].append(sids)
            k = drow[str(names[f])]; dd = z["dets"][z["starts"][k]:z["starts"][k + 1]]
            fi = frame_inputs(sids, dd)
            S["out"][LIGHT]["score"].append(o["score"][a:b]); S["out"][LIGHT]["detf"].append(fi["detf"]); S["out"][LIGHT]["occ"].append(fi["occ"])
            r9 = ff[str(names[f])]
            S["B"].append(float(r9["brightness"]))
            S["M"].append(np.nan if r9["motion_raw"] in ("", "None") else float(sf.scale(float(r9["motion_raw"]), SCALES["motion"])))
        S["B"], S["M"] = np.array(S["B"]), np.array(S["M"])
        seqs.append(S)
    return seqs

def signals_of(seq):
    out = rp.run_sequence(seq, "fixed_n", {}, CTX)
    F = out["F"]
    return {"brightness": F[:, 0], "confidence": F[:, 2], "density": F[:, 3], "occlusion": F[:, 4], "instability": F[:, 5],
            "difficulty": out["D"], "slot_risk": out["Rg"]}

t0 = time.time()
try:
    SEQ = {s: build(s) for s in ("val", "test")}
    for s in SEQ:
        for q in SEQ[s]:
            q["sig"] = signals_of(q)
except Exception as e:
    write_status(STATUS, "12_drift_monitoring", RUN_ID, "FAILED", CFG_HASH, repr(e)[:500]); raise
print("clean signals: {} val / {} test sequences, {} / {} frames, {:.0f} s".format(len(SEQ["val"]), len(SEQ["test"]),
      sum(len(q["sec"]) for q in SEQ["val"]), sum(len(q["sec"]) for q in SEQ["test"]), time.time() - t0))

def base_from(seqs):
    return {s: dm.baseline_quantiles(np.concatenate([q["sig"][s] for q in seqs])) for s in dm.SIGNALS} if seqs else None
BASE = {c: base_from([q for q in SEQ["val"] if q["camera"] == c]) for c in sorted({q["camera"] for q in SEQ["val"]})}
BASE_LOO = {}
for c, d in sorted({(q["camera"], q["day"]) for q in SEQ["val"]}):
    BASE_LOO[(c, d)] = base_from([q for q in SEQ["val"] if q["camera"] == c and q["day"] != d])
def baseline_for(split, q):
    return BASE_LOO.get((q["camera"], q["day"])) if split == "val" else BASE.get(q["camera"])

def window_scores(sig, base, test):
    T = len(sig[dm.SIGNALS[0]])
    return [dm.window_scores(test, base, {s: sig[s][t - WINDOW + 1:t + 1] for s in dm.SIGNALS}) for t in range(WINDOW - 1, T)]

CLEAN, THRS = {}, {}
for test in dm.TESTS:
    CLEAN[test] = [w for q in SEQ["val"] if baseline_for("val", q) for w in window_scores(q["sig"], baseline_for("val", q), test)]
    THRS[test] = dm.calibrate(CLEAN[test], WARN_RATE, CRIT_RATE)
    print("\n{} thresholds from {} clean val windows (leave-own-day-out baselines): critical overall {:.2f} x warning".format(
        test.upper(), len(CLEAN[test]), THRS[test]["critical_overall"]))
    print("  warning level per signal:", {s: round(x, 3) for s, x in THRS[test]["warning"].items()})
    sev = Counter(dm.severity(w, THRS[test]) for w in CLEAN[test])
    print("  clean val windows: ", {k: round(v / max(1, len(CLEAN[test])), 4) for k, v in sev.items()})
missing = sorted({q["camera"] for q in SEQ["test"]} - set(BASE))
print("\ntest cameras without a val baseline (not monitored):", missing or "none")

In [ ]:
# 4. Synthetic drift: perturb the second half of sampled sequences, re-detect with YOLO11n, monitor, measure real accuracy
from ultralytics import YOLO
MODEL = YOLO(str(WEIGHTS))
VARIANTS = pt.variants()

def pick(split):
    c = [i for i, q in enumerate(SEQ[split]) if len(q["sec"]) >= MIN_LEN and baseline_for(split, q)]
    c = sorted(c, key=lambda i: (SEQ[split][i]["site"], i))
    n = min(2 if SMOKE else N_SYNTH, len(c))
    return [c[i] for i in np.linspace(0, len(c) - 1, n).astype(int)] if n else []

def detect(arrays):
    out = []
    for r in MODEL.predict(arrays, conf=0.05, iou=0.7, imgsz=IMGSZ, device=DEVICE, half=DEVICE != "cpu", batch=16, max_det=300, verbose=False, stream=True):
        b = r.boxes
        out.append(np.concatenate([b.xyxy.cpu().numpy(), b.conf.cpu().numpy()[:, None]], 1).astype(np.float32) if len(b) else np.zeros((0, 5), np.float32))
    return out

def run_variants(split, q):
    """Per frame and variant: slot inputs, darkness B and the small gray image (for motion)."""
    T = len(q["sec"]); res = {v[0]: {"fi": [None] * T, "B": np.zeros(T), "gray": [None] * T} for v in VARIANTS}
    for c0 in range(0, T, CHUNK_FRAMES):
        ts = list(range(c0, min(T, c0 + CHUNK_FRAMES)))
        ims = []
        for t in ts:
            with Image.open(str(EXPORT / "images" / split / q["names"][t])) as im:
                ims.append(im.convert("RGB"))
        for name, kind, val in VARIANTS:
            pims = [pt.apply(im, kind, val, seed=int(q["image_ids"][t])) for im, t in zip(ims, ts)]
            dets = detect([np.asarray(p)[:, :, ::-1] for p in pims])
            for t, p, d in zip(ts, pims, dets):
                g = sf.to_gray_small(p, REGION[q["camera"]])
                res[name]["fi"][t] = frame_inputs(q["slot_ids"][t], d); res[name]["B"][t] = sf.brightness(g); res[name]["gray"][t] = g
        del ims
    return res

def compose(q, res, name, onset):
    """Frames before onset from the clean re-detection, from onset on from the variant (motion across the boundary included)."""
    T = len(q["sec"]); src = lambda t: res["none" if t < onset else name]
    S = dict(q, out={LIGHT: {k: [src(t)["fi"][t][k] for t in range(T)] for k in ("score", "detf", "occ")}},
             B=np.array([src(t)["B"][t] for t in range(T)]))
    M = [np.nan]
    for t in range(1, T):
        m = sf.motion_raw(src(t)["gray"][t], src(t - 1)["gray"][t - 1]); M.append(np.nan if m is None else float(sf.scale(m, SCALES["motion"])))
    S["M"] = np.array(M)
    return S

def post_accuracy(q, S, onset):
    right = n = fv = fo = 0
    for t in range(onset, len(q["sec"])):
        v = q["valid"][t]; g = q["gt"][t][v].astype(bool); p = S["out"][LIGHT]["score"][t][v] >= THR
        right += int((p == g).sum()); n += len(g); fv += int((g & ~p).sum()); fo += int((~g & p).sum())
    return right / max(1, n), fv, fo

SYN, EVENTS, AGREE = [], {t: [] for t in dm.TESTS}, []
t0 = time.time()
try:
    for split in ("val", "test"):
        for i in pick(split):
            q = SEQ[split][i]; onset = len(q["sec"]) // 2; base = baseline_for(split, q)
            res = run_variants(split, q)
            for t in range(len(q["sec"])):                    # re-detection of the clean frames vs the cached notebook 07 detections
                v = q["valid"][t]
                AGREE.append(float(((res["none"]["fi"][t]["score"][v] >= THR) == (q["out"][LIGHT]["score"][t][v] >= THR)).mean()) if v.any() else 1.0)
            acc0 = None
            for name, kind, val in VARIANTS:
                S = compose(q, res, name, onset); sig = signals_of(S)
                acc, fv, fo = post_accuracy(q, S, onset)
                acc0 = acc if name == "none" else acc0
                row = {"split": split, "seq": i, "camera": q["camera"], "site": q["site"], "variant": name, "kind": kind or "none",
                       "level": "none" if kind is None else name.split("_")[1], "frames": len(q["sec"]), "onset": onset,
                       "post_accuracy": round(acc, 4), "accuracy_drop": round(acc0 - acc, 4), "post_FV": fv, "post_FO": fo}
                for test in dm.TESTS:
                    mon = dm.DriftMonitor(q["camera"], base, THRS[test], test, WINDOW)
                    sev, ov = [], []
                    for t in range(len(q["sec"])):
                        r = mon.update({s: sig[s][t] for s in dm.SIGNALS}, frame=q["names"][t], timestamp=float(q["sec"][t]))
                        sev.append(r["severity"]); ov.append(r["overall"])
                    for e in mon.history:
                        e.update(split=split, variant=name, onset_frame=q["names"][onset])
                    EVENTS[test] += mon.history
                    pre = sev[WINDOW - 1:onset]                       # windows entirely before onset
                    full = sev[onset + WINDOW - 1:]                    # windows entirely after onset
                    post = [k for k in range(onset, len(sev)) if sev[k] != "ok"]
                    row.update({test + "_pre_rate": round(float(np.mean([x != "ok" for x in pre])), 3) if pre else None,
                                test + "_post_rate": round(float(np.mean([x != "ok" for x in full])), 3) if full else None,
                                test + "_post_critical": round(float(np.mean([x == "critical" for x in full])), 3) if full else None,
                                test + "_delay": (post[0] - onset) if post else None,
                                test + "_max_overall": round(float(np.nanmax(ov[onset:])), 3) if not np.isnan(ov[onset:]).all() else None,
                                test + "_overall": [None if np.isnan(x) else round(float(x), 3) for x in ov]})
                SYN.append(row)
            print("{} seq {} ({}, {} frames): done, {:.0f} s".format(split, i, q["camera"], len(q["sec"]), time.time() - t0))
            del res; gc.collect()
except Exception as e:
    write_status(STATUS, "12_drift_monitoring", RUN_ID, "FAILED", CFG_HASH, repr(e)[:500]); raise
print("re-detection of clean frames agrees with the cached notebook 07 slot decisions on {:.4f} of slots (mean over frames)".format(np.mean(AGREE) if AGREE else float("nan")))

def summary(split, test):
    rows = []
    for name, kind, val in VARIANTS:
        r = [x for x in SYN if x["split"] == split and x["variant"] == name]
        if not r:
            continue
        d = [x[test + "_delay"] for x in r if x[test + "_delay"] is not None]
        m = lambda k: round(float(np.mean([x[test + k] for x in r if x[test + k] is not None])), 3) if any(x[test + k] is not None for x in r) else None
        rows.append({"variant": name, "sequences": len(r), "alert_rate_drifted": m("_post_rate"), "critical_rate_drifted": m("_post_critical"),
                     "alert_rate_before_onset": m("_pre_rate"), "mean_first_alert_delay_frames": round(float(np.mean(d)), 2) if d else None,
                     "accuracy_drop": round(float(np.mean([x["accuracy_drop"] for x in r])), 4), "post_FV": sum(x["post_FV"] for x in r)})
    return rows

def det_rate(split, test):
    r = [x for x in SYN if x["split"] == split and x["variant"] != "none" and x[test + "_post_rate"] is not None]
    d = [x[test + "_delay"] for x in r if x[test + "_delay"] is not None]
    return (round(float(np.mean([x[test + "_post_rate"] for x in r])), 4) if r else 0.0, -(np.mean(d) if d else 1e9))
TEST_CHOSEN = max(dm.TESTS, key=lambda t: (det_rate("val", t), t == "psi"))
print("chosen on val:", TEST_CHOSEN.upper(), "| val alert rate on drifted windows (all perturbations)", {t: det_rate("val", t)[0] for t in dm.TESTS})
H = "  {:<14} {:>4} {:>11} {:>11} {:>12} {:>9} {:>10} {:>8}"
for split in ("val", "test"):
    for test in dm.TESTS:
        print("\n{} synthetic drift, {}{}  (alert rates per window; drifted = window entirely after onset; accuracy drop = clean minus perturbed, post-onset frames)".format(
            split.upper(), test.upper(), "  <- chosen" if test == TEST_CHOSEN else ""))
        print(H.format("variant", "seqs", "alert drift", "crit drift", "alert before", "1st delay", "acc drop", "post FV"))
        for r in summary(split, test):
            print(H.format(r["variant"], r["sequences"], str(r["alert_rate_drifted"]), str(r["critical_rate_drifted"]), str(r["alert_rate_before_onset"]),
                           str(r["mean_first_alert_delay_frames"]), r["accuracy_drop"], r["post_FV"]))

In [ ]:
# 5. Natural drift: every test sequence (no perturbation) against its camera's val baseline, chosen test
NAT, NAT_EVENTS = [], []
for q in SEQ["test"]:
    base = BASE.get(q["camera"])
    if not base:
        continue
    mon = dm.DriftMonitor(q["camera"], base, THRS[TEST_CHOSEN], TEST_CHOSEN, WINDOW)
    for t in range(len(q["sec"])):
        r = mon.update({s: q["sig"][s][t] for s in dm.SIGNALS}, frame=q["names"][t], timestamp=float(q["sec"][t]))
        if r["window_full"]:
            NAT.append({"site": q["site"], "camera": q["camera"], "climate": q["climate"][t], "severity": r["severity"],
                        "top": max(dm.SIGNALS, key=lambda s: -1 if np.isnan(r["scores"][s]) else r["scores"][s] / THRS[TEST_CHOSEN]["warning"][s])})
    NAT_EVENTS += mon.history
def rates(key):
    out = {}
    for g in sorted({r[key] for r in NAT}):
        rr = [r for r in NAT if r[key] == g]
        out[g] = {"windows": len(rr), "warning_or_worse": round(float(np.mean([r["severity"] != "ok" for r in rr])), 3),
                  "critical": round(float(np.mean([r["severity"] == "critical" for r in rr])), 3)}
    return out
print("TEST natural drift ({} windows; clean val windows warn at about {:.0%} by construction)".format(len(NAT), WARN_RATE))
print("  all:", {"warning_or_worse": round(float(np.mean([r["severity"] != "ok" for r in NAT])), 3) if NAT else None,
                 "critical": round(float(np.mean([r["severity"] == "critical" for r in NAT])), 3) if NAT else None})
for key in ("site", "climate"):
    print("  by {}:".format(key))
    for g, v in rates(key).items():
        print("    {:<14} {}".format(g, v))
print("  signal most often responsible (alerting windows):", Counter(r["top"] for r in NAT if r["severity"] != "ok").most_common())
print("\nexample alert event:")
ex = (NAT_EVENTS or EVENTS[TEST_CHOSEN])[:1]
print(json.dumps([{k: (v if k != "scores" else {s: None if np.isnan(x) else round(x, 3) for s, x in v.items()}) for k, v in e.items()} for e in ex], indent=1, default=str))

In [ ]:
# Plots: detection vs real accuracy drop, score timelines around onset, baseline vs drifted window
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(15, 4.6))
S_ = summary("test", TEST_CHOSEN)
for r in S_:
    y = np.nan if r["alert_rate_drifted"] is None else r["alert_rate_drifted"]
    ax[0].scatter(r["accuracy_drop"], y, s=40)
    ax[0].annotate(r["variant"], (r["accuracy_drop"], y), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax[0].set(xlabel="slot accuracy drop after onset (clean - perturbed)", ylabel="alert rate of fully drifted windows",
          title="TEST: alerts vs real degradation ({})".format(TEST_CHOSEN.upper())); ax[0].grid(alpha=0.3)
rows = [x for x in SYN if x["split"] == "test"]
if rows:
    seq0 = rows[0]["seq"]
    for x in rows:
        if x["seq"] == seq0 and x["variant"] in ("none", "dark_strong", "blur_strong", "haze_strong", "shift_strong", "noise_strong"):
            ax[1].plot([np.nan if v is None else v for v in x[TEST_CHOSEN + "_overall"]], label=x["variant"])
    ax[1].axvline(rows[0]["onset"], color="k", ls=":"); ax[1].axhline(1.0, color="orange", lw=0.8); ax[1].axhline(THRS[TEST_CHOSEN]["critical_overall"], color="r", lw=0.8)
    ax[1].set(xlabel="frame", ylabel="overall drift score (1 = warning)", title="TEST {}: drift score (dotted: onset)".format(rows[0]["camera"])); ax[1].legend(fontsize=7); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(str(OUT / "drift_detection{}.png".format(SUF)), dpi=110); plt.show()

# baseline vs current: the camera baseline (CDF of its quantiles) and the clean signal values of one test sequence
if SEQ["test"] and BASE.get(SEQ["test"][0]["camera"]):
    q = SEQ["test"][0]; base = BASE[q["camera"]]
    fig, ax = plt.subplots(1, len(dm.SIGNALS), figsize=(3 * len(dm.SIGNALS), 2.8))
    for a, s in zip(ax, dm.SIGNALS):
        a.plot(base[s], np.linspace(0, 1, 101), label="val baseline")
        x = np.sort(q["sig"][s][~np.isnan(q["sig"][s])])
        if len(x):
            a.step(x, np.arange(1, len(x) + 1) / len(x), where="post", label="test day")
        a.set_title(s, fontsize=9); a.grid(alpha=0.3)
    ax[0].legend(fontsize=7); plt.suptitle("{} {}: baseline vs current distribution (CDF)".format(q["camera"], q["day"]), fontsize=10)
    plt.tight_layout(); plt.savefig(str(OUT / "baseline_vs_current{}.png".format(SUF)), dpi=110); plt.show()

In [ ]:
# Record, baselines for the Orin, alert history, status
from src.common.env_record import save_environment
def jclean(x):
    if isinstance(x, dict):
        return {str(k): jclean(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [jclean(v) for v in x]
    if isinstance(x, (float, np.floating)):
        return None if np.isnan(x) else float(x)
    if isinstance(x, np.ndarray):
        return jclean(x.tolist())
    if isinstance(x, (np.integer, np.bool_)):
        return x.item()
    return x
(OUT / "baselines{}.json".format(SUF)).write_text(json.dumps(jclean({c: {s: b[s] for s in dm.SIGNALS} for c, b in BASE.items() if b})))
(OUT / "alert_history_test{}.json".format(SUF)).write_text(json.dumps(jclean({"natural": NAT_EVENTS, "synthetic": [e for e in EVENTS[TEST_CHOSEN] if e.get("split") == "test"]}), indent=1))
RECORD = {"run_id": RUN_ID, "config_hash": CFG_HASH, "smoke_test": SMOKE, "dataset_split_hash": SPLIT_INFO["manifest_hash"],
          "notebook07_run_id": st07["run_id"], "notebook09_run_id": R09["run_id"], "signals": list(dm.SIGNALS), "window_frames": WINDOW,
          "test_chosen_on_val": TEST_CHOSEN, "thresholds": THRS, "warn_rate": WARN_RATE, "crit_rate": CRIT_RATE,
          "selection_rule": "val synthetic drift: highest alert rate on fully drifted windows (false-alarm rate fixed by the clean-window thresholds), then shortest first-alert delay, then PSI",
          "redetection_agreement_with_nb07": float(np.mean(AGREE)) if AGREE else None,
          "synthetic": {s: {t: summary(s, t) for t in dm.TESTS} for s in ("val", "test")},
          "synthetic_rows": [{k: v for k, v in r.items() if not k.endswith("_overall")} for r in SYN],
          "natural_test": {"by_site": rates("site"), "by_climate": rates("climate")},
          "policy": "alerts for human review only; no automatic retraining; distribution shift is not proof of accuracy loss",
          "limitation": "synthetic perturbations are stand-ins for real drift; frames 5 min / 30 min apart; window in frames"}
(OUT / "drift_record{}.json".format(SUF)).write_text(json.dumps(jclean(RECORD), indent=2))
save_environment(str(OUT / "environment.json"))
write_status(STATUS, "12_drift_monitoring", RUN_ID, "COMPLETE", CFG_HASH, "smoke={}".format(SMOKE),
             extra=jclean({"test": TEST_CHOSEN, "test_alert_rate_drifted": {r["variant"]: r["alert_rate_drifted"] for r in summary("test", TEST_CHOSEN)}}))
print("COMPLETE | saved to", OUT, sorted(p.name for p in OUT.iterdir()))

## What to send back
Download the notebook after **Save & Run All** (SMOKE = False) and send it.
`artifacts/drift/` (per-camera baselines, thresholds, alert history) feeds the API and the dashboard's Drift & Alerts page, so **Save Version** with outputs.